In [2]:
%pip install seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 12.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [2]:
cols = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'num'
]

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

df = pd.read_csv(url, names=cols, na_values='?')

print('Loaded dataframe with shape:', df.shape)

df.head()

Loaded dataframe with shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


- 303 rows: 303 patient records.
- 14 columns: 13 patient characteristics and 1 heart disease outcome.
Each row represents one patient, and each column contains information about that patient.

| Column | Meaning | 
|---|---|---|
| `age` | Patient's age |
| `sex` | Biological sex |
| `cp` | Chest pain type |
| `trestbps` | Resting blood pressure 
| `chol` | Serum cholesterol | 
| `fbs` | Fasting blood sugar >120 mg/dL
| `restecg` | Resting ECG findings |
| `thalach` | Maximum heart rate achieved
| `exang` | Exercise-induced angina 
| `oldpeak` | ST depression induced by exercise relative to rest 
| `slope` | Slope of the peak exercise ST segment
| `ca` | Number of major vessels coloured by fluoroscopy 
| `thal` | Thallium stress test result 
| **`num`** | **Heart disease outcome** 
Our machine-learning model will use the first 13 features to predict the heart disease outcome.

In [3]:
df.info()

df.describe().T

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(13), int64(1)
memory usage: 33.3 KB


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


- 303 patients and 14 columns.
- Average patient age: 54.4 years.
- Average resting blood pressure: 131.7 mmHg.
- Average cholesterol: 246.7 mg/dL.
- Average maximum heart rate: 149.6 bpm.
Missing values:
Column	Missing values
ca	4
thal	2
All other columns	0

These missing values will be handled using imputation in our preprocessing pipeline.

In [4]:
df["target"] = (df["num"] > 0).astype(int)

df = df.drop(columns=["num"])

print(df["target"].value_counts())

target
0    164
1    139
Name: count, dtype: int64


0 = No heart disease
1 = Heart disease present (original values 1–4)

164 patients (54.1%) without recorded heart disease.
139 patients (45.9%) with recorded heart disease.

In [5]:
cat_cols = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "thal"
]

num_cols = [
    c for c in df.columns
    if c not in cat_cols + ["target"]
]

print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

Numeric cols: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Categorical cols: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']


Handle missing values, scale numerical features and convert categorical features into numbers.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

print("Preprocessing pipeline created successfully!")

Preprocessing pipeline created successfully!


In [7]:
X = df.drop(columns=["target"]).copy()
y = df["target"].copy()

X_pre = preprocessor.fit_transform(X)

print("Preprocessed shape:", X_pre.shape)

Preprocessed shape: (303, 25)


Group patients into age categories to help the model identify age-related patterns.

In [8]:
X2 = X.copy()

X2["age_bucket"] = pd.cut(
    X2["age"],
    bins=[0, 40, 55, 70, 120],
    labels=["<40", "40-55", "55-70", "70+"]
)

X2.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,age_bucket
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,55-70
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,55-70
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,55-70
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,<40
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,40-55


Prepare patient data for training Logistic Regression and Decision Tree models.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (242, 13)
Testing data: (61, 13)


242 patients (80%) for training.
61 patients (20%) for testing.

Train two different machine-learning algorithms to predict heart disease.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline

pipe_lr = make_pipeline(
    preprocessor,
    LogisticRegression(max_iter=1000)
)

pipe_dt = make_pipeline(
    preprocessor,
    DecisionTreeClassifier(max_depth=5, random_state=42)
)

pipe_lr.fit(X_train, y_train)
pipe_dt.fit(X_train, y_train)

print("Both models trained successfully!")

Both models trained successfully!


Compare Logistic Regression and Decision Tree using accuracy, precision, recall, F1-score and prediction time.

In [11]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)
import time

for name, model in [('Logistic', pipe_lr), ('DecisionTree', pipe_dt)]:
    t0 = time.time()
    preds = model.predict(X_test)
    t1 = time.time()

    print(f"--- {name} ---")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Precision:", precision_score(y_test, preds))
    print("Recall:", recall_score(y_test, preds))
    print("F1:", f1_score(y_test, preds))
    print("Inference time (seconds):", t1 - t0)

--- Logistic ---
Accuracy: 0.8688524590163934
Precision: 0.8125
Recall: 0.9285714285714286
F1: 0.8666666666666667
Inference time (seconds): 0.01812434196472168
--- DecisionTree ---
Accuracy: 0.7704918032786885
Precision: 0.71875
Recall: 0.8214285714285714
F1: 0.7666666666666667
Inference time (seconds): 0.005888700485229492


Accuracy: Percentage of correct predictions.
Precision: Of patients predicted to have heart disease, how many actually have it?
Recall: Of patients with heart disease, how many did the model identify?
F1-score: Balance between precision and recall.
Inference time: How quickly the model makes predictions.

Logistic Regression performed better across all four classification metrics. It identified approximately 93% of patients with recorded heart disease in this test set.

The Decision Tree was faster, but less accurate. These results are based on only 61 test patients, so they don't establish clinical reliability.

Learn ensemble learning, where multiple decision trees work together to make predictions.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 8, 12, None],
    "max_features": ["sqrt", "log2", 0.5]
}

print("Random Forest and parameters ready!")

Random Forest and parameters ready!


Automatically test different Random Forest settings to find the best-performing combination.

In [13]:
rs = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=8,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    random_state=42
)

rs.fit(X_train, y_train)

print("Best parameters:", rs.best_params_)

Best parameters: {'n_estimators': 200, 'max_features': 0.5, 'max_depth': 8}


Parameter	Best value	Simple explanation
n_estimators	200	The Random Forest uses 200 decision trees to make predictions.
max_features	0.5	Each split considers 50% of the available features.
max_depth	8	Each decision tree can grow to a maximum depth of 8 levels.

Instead of manually choosing Random Forest settings, RandomizedSearchCV tested 8 combinations using 3-fold cross-validation and selected the combination with the highest average F1-score.

This process is called hyperparameter tuning.

Check how accurately your tuned Random Forest predicts heart disease on the test dataset.

In [14]:
best_rf = rs.best_estimator_

preds = rs.predict(X_test)

print("Test F1:", f1_score(y_test, preds))
print("Test Accuracy:", accuracy_score(y_test, preds))

Test F1: 0.8666666666666667
Test Accuracy: 0.8688524590163934


Model	Accuracy	F1-score
Logistic Regression	86.89%	86.67%
Decision Tree	77.05%	76.67%
Random Forest	86.89%	86.67%

Conclusion: Logistic Regression and Random Forest achieved identical accuracy and F1-scores on your test dataset.

Logistic Regression may be preferable for interpretability and simplicity, but we need further evaluation before choosing a model for clinical use.

Check whether the Random Forest performs differently across patient sex groups.

F1-score tells us how well a machine-learning model balances precision and recall.
Let's understand with a medical example ❤️
Imagine your AI model predicts whether patients have heart disease.
- Precision: Of the patients the model predicts have heart disease, how many actually have it?
- Recall: Of all patients who actually have heart disease, how many does the model correctly identify?
- F1-score: Combines precision and recall into one score.
The formula is:
\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

Why are we comparing F1-scores?
Because accuracy alone can be misleading.

For example, imagine 100 patients, but only 10 have heart disease. A model that predicts everyone is healthy would have 90% accuracy, yet miss every patient with heart disease.

F1 helps us identify models that balance finding positive cases with avoiding false alarms.

In [15]:
res = X_test.reset_index(drop=True).copy()

res["y_true"] = y_test.to_numpy()
res["y_pred"] = preds

for gname, gdf in res.groupby("sex"):
    print("Group sex =", gname)
    print("Count:", len(gdf))
    print("F1:", f1_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("Recall:", recall_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("Precision:", precision_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("---")

Group sex = 0.0
Count: 20
F1: 0.9230769230769231
Recall: 0.8571428571428571
Precision: 1.0
---
Group sex = 1.0
Count: 41
F1: 0.851063829787234
Recall: 0.9523809523809523
Precision: 0.7692307692307693
---


Female patients: The model correctly identified approximately 86% of patients with recorded heart disease. Its precision was 100%, meaning none of its positive predictions were false positives in this small test group.
Male patients: The model identified approximately 95% of patients with recorded heart disease, but had more false-positive predictions.These differences do not prove that the model is biased. We have only 20 female and 41 male patients in the test set, so the estimates are uncertain.

Check whether the model performs differently in patients younger than 55 versus those aged 55 and above

In [16]:
res["age_bucket"] = (res["age"] >= 55).astype(int)

for gname, gdf in res.groupby("age_bucket"):
    print("Age >=55?", gname)
    print("Count:", len(gdf))
    print("F1:", f1_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("Recall:", recall_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("Precision:", precision_score(gdf["y_true"], gdf["y_pred"], zero_division=0))
    print("---")

Age >=55? 0
Count: 28
F1: 0.9333333333333333
Recall: 1.0
Precision: 0.875
---
Age >=55? 1
Count: 33
F1: 0.8444444444444444
Recall: 0.9047619047619048
Precision: 0.7916666666666666
---


- Under 55: The model identified every patient with recorded heart disease in this test group (100% recall).
- 55 and older: The model identified approximately 90% of patients with recorded heart disease.
- Conclusion: The model performed better in the younger group on this test set.
However, these groups contain only 28 and 33 patients. The differences are not enough to establish age-related bias or clinical reliability.

Breast Cancer Classification Exercise

Compare Logistic Regression, Decision Tree, Random Forest and XGBoost using another medical dataset.

In [17]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

df_bc = data.frame

df_bc.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


Target 0: Malignant (cancerous)
Target 1: Benign (non-cancerous)

Separate the input features from the target and split the data for training and testing.

In [18]:
X_bc = df_bc.drop(columns=["target"])
y_bc = df_bc["target"]

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc,
    y_bc,
    test_size=0.2,
    stratify=y_bc,
    random_state=42
)

print("Training:", X_train_bc.shape)
print("Testing:", X_test_bc.shape)

Training: (455, 30)
Testing: (114, 30)


455 samples (80%) for training.
114 samples (20%) for testing.
30 features describing cell nuclei

In [19]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model_lr_bc = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model_lr_bc.fit(X_train_bc, y_train_bc)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


It standardises the numerical features and trains a model to distinguish benign from malignant samples.

Measure how accurately your model classifies breast cancer samples.

In [20]:
pred_lr_bc = model_lr_bc.predict(X_test_bc)

print("Accuracy:", accuracy_score(y_test_bc, pred_lr_bc))
print("Precision:", precision_score(y_test_bc, pred_lr_bc))
print("Recall:", recall_score(y_test_bc, pred_lr_bc))
print("F1-score:", f1_score(y_test_bc, pred_lr_bc))

Accuracy: 0.9824561403508771
Precision: 0.9861111111111112
Recall: 0.9861111111111112
F1-score: 0.9861111111111112


model correctly classified 112 out of 114 samples.

In [21]:
from sklearn.tree import DecisionTreeClassifier

model_dt_bc = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_dt_bc.fit(X_train_bc, y_train_bc)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


Measure the Decision Tree's accuracy, precision, recall and F1-score so we can compare it with Logistic Regression.

In [22]:
pred_dt_bc = model_dt_bc.predict(X_test_bc)

print("Accuracy:", accuracy_score(y_test_bc, pred_dt_bc))
print("Precision:", precision_score(y_test_bc, pred_dt_bc))
print("Recall:", recall_score(y_test_bc, pred_dt_bc))
print("F1-score:", f1_score(y_test_bc, pred_dt_bc))

Accuracy: 0.9210526315789473
Precision: 0.9565217391304348
Recall: 0.9166666666666666
F1-score: 0.9361702127659575


Logistic Regression performed better across all four metrics on your test dataset.

Remember, these precision, recall and F1-scores measure the benign class. In medical AI, we must also assess how many malignant cases are missed.

Learn how combining multiple decision trees can improve classification performance.

In [23]:
from sklearn.ensemble import RandomForestClassifier

model_rf_bc = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model_rf_bc.fit(X_train_bc, y_train_bc)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [24]:
pred_rf_bc = model_rf_bc.predict(X_test_bc)

print("Accuracy:", accuracy_score(y_test_bc, pred_rf_bc))
print("Precision:", precision_score(y_test_bc, pred_rf_bc))
print("Recall:", recall_score(y_test_bc, pred_rf_bc))
print("F1-score:", f1_score(y_test_bc, pred_rf_bc))

Accuracy: 0.956140350877193
Precision: 0.958904109589041
Recall: 0.9722222222222222
F1-score: 0.9655172413793104


Metric	Logistic Regression	Decision Tree	Random Forest
Accuracy	98.25%	92.11%	95.61%
Precision	98.61%	95.65%	95.89%
Recall	98.61%	91.67%	97.22%
F1-score	98.61%	93.62%	96.55%

Logistic Regression is currently your best-performing model on this test set.

These metrics evaluate the benign class, so we'll also need to assess malignant-case detection before making a final model recommendation.

Prepare XGBoost, a boosting algorithm that builds decision trees sequentially to improve predictions.

In [25]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 11.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 MB 3.2 MB/s eta 0:00:00:00:0100:01
Note: you may need to restart the kernel to use updated packages.


Train our fourth breast cancer classifier using gradient boosting.

In [26]:
from xgboost import XGBClassifier

model_xgb_bc = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

model_xgb_bc.fit(X_train_bc, y_train_bc)

print("XGBoost trained successfully!")

XGBoost trained successfully!


Measure XGBoost's accuracy, precision, recall and F1-score to compare it with your other three models.

In [27]:
pred_xgb_bc = model_xgb_bc.predict(X_test_bc)

print("Accuracy:", accuracy_score(y_test_bc, pred_xgb_bc))
print("Precision:", precision_score(y_test_bc, pred_xgb_bc))
print("Recall:", recall_score(y_test_bc, pred_xgb_bc))
print("F1-score:", f1_score(y_test_bc, pred_xgb_bc))

Accuracy: 0.9473684210526315
Precision: 0.9459459459459459
Recall: 0.9722222222222222
F1-score: 0.958904109589041


Model	Accuracy	F1-score
Logistic Regression	98.25%	98.61%
Decision Tree	92.11%	93.62%
Random Forest	95.61%	96.55%
XGBoost	94.74%	95.89%

Conclusion: Logistic Regression performed best on your test dataset, despite being simpler than Random Forest and XGBoost.

Cross-Validation
Evaluate model performance across multiple data splits instead of relying on one train/test split.

In [28]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model_lr_bc,
    X_train_bc,
    y_train_bc,
    cv=5,
    scoring="accuracy"
)

print("Cross-validation scores:", cv_scores)
print("Mean accuracy:", cv_scores.mean())

Cross-validation scores: [0.96703297 0.97802198 0.96703297 1.         0.98901099]
Mean accuracy: 0.9802197802197803


Your Logistic Regression model achieved 98.02% mean accuracy using 5-fold cross-validation.
This suggests the model performs consistently across different training and validation splits.

Compare All Four Models Using Cross-Validation
Purpose: Determine which model performs most consistently across five folds.

In [29]:
models = {
    "Logistic Regression": model_lr_bc,
    "Decision Tree": model_dt_bc,
    "Random Forest": model_rf_bc,
    "XGBoost": model_xgb_bc
}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_bc,
        y_train_bc,
        cv=5,
        scoring="accuracy"
    )

    print(f"{name}: {scores.mean():.4f}")

Logistic Regression: 0.9802
Decision Tree: 0.9319
Random Forest: 0.9538
XGBoost: 0.9714


Model	Cross-validation accuracy
Logistic Regression	98.02%
XGBoost	97.14%
Random Forest	95.38%
Decision Tree	93.19%

Conclusion: Logistic Regression performed best in your experiment. It is also relatively simple, fast and interpretable.

For your Week 5 production-selection question, Logistic Regression is the strongest candidate, although clinical deployment would require further validation, including malignant-case recall, calibration, fairness and external testing.

Evaluate Malignant Cancer Detection
Purpose: Check how well your best model identifies malignant cases rather than just benign cases.

In [30]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_bc,
    pred_lr_bc,
    target_names=["Malignant", "Benign"]
))

              precision    recall  f1-score   support

   Malignant       0.98      0.98      0.98        42
      Benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



our Logistic Regression model performed very well on the breast cancer test dataset.
Step 41 — Understand Your Results
Metric	Malignant	Benign
Precision	98%	99%
Recall	98%	99%
F1-score	98%	99%
Samples	42	72

Malignant recall (98%): The model identified approximately 98% of malignant samples.
Benign recall (99%): The model correctly identified approximately 99% of benign samples.
Overall accuracy (98.25%): It correctly classified 112 out of 114 samples.

Which model would you push to production, and why?

Answer: I would select Logistic Regression as the candidate for further validation because it achieved the highest test accuracy (98.25%) and cross-validation accuracy (98.02%). It is also relatively fast and interpretable compared with more complex ensemble models.

However, before clinical deployment, I would evaluate its performance on independent hospital data, examine fairness across patient groups and assess the consequences of missed malignant cases.